# QFF2026 QML Track v3 — Phase 2 submission

**Selected problem:** PS-Q2 — high-dimensional quantum-kernel classification  
**Required common tasks:** Task A — Bell-state fidelity; Task B — Processor D QFT-3

This notebook is the Phase 2 entry point. It uses all 800 labeled PS-Q2 rows
and all 50 features, performs leakage-safe five-fold model selection, writes
the 200 required test predictions, and validates them with the official kit
validator. Test labels are unavailable and are never loaded.

Run the notebook from the repository root in a dedicated Python 3.11–3.13
environment installed from `qml-requirements.txt`. Any pin mismatch, official
benchmark mismatch, experiment failure, or prediction-validation error aborts
execution.

In [1]:
import json
import subprocess
import sys
from importlib import metadata
from pathlib import Path

ROOT = Path.cwd().resolve()
REQUIRED_FILES = (
    "qml-requirements.txt",
    "processor_d_config.json",
    "q2_solution.py",
)
missing_files = [name for name in REQUIRED_FILES if not (ROOT / name).is_file()]
if missing_files:
    raise FileNotFoundError(
        f"Run this notebook from the repository root; missing: {missing_files}"
    )
if not ((3, 11) <= sys.version_info[:2] <= (3, 13)):
    raise RuntimeError(
        f"QML kit v3 supports Python 3.11–3.13; found {sys.version.split()[0]}"
    )

pin_file = ROOT / "qml-requirements.txt"
required_versions = {}
for raw_line in pin_file.read_text(encoding="utf-8").splitlines():
    requirement = raw_line.strip()
    if not requirement or requirement.startswith("#"):
        continue
    if requirement.count("==") != 1:
        raise ValueError(f"Every direct QML requirement must be pinned: {requirement}")
    distribution, pinned_version = requirement.split("==", 1)
    required_versions[distribution] = pinned_version

version_rows = []
version_problems = []
for distribution, pinned_version in required_versions.items():
    try:
        installed_version = metadata.version(distribution)
    except metadata.PackageNotFoundError:
        installed_version = "NOT INSTALLED"
    status = "OK" if installed_version == pinned_version else "MISMATCH"
    version_rows.append((distribution, pinned_version, installed_version, status))
    if status != "OK":
        version_problems.append(
            f"{distribution}: required {pinned_version}, found {installed_version}"
        )

print(f"Python {sys.version.split()[0]}")
print(f"{'package':<24} {'required':<12} {'installed':<16} status")
for distribution, pinned_version, installed_version, status in version_rows:
    print(f"{distribution:<24} {pinned_version:<12} {installed_version:<16} {status}")
if version_problems:
    raise RuntimeError(
        "Pinned package check failed:\n  - " + "\n  - ".join(version_problems)
    )

PHASE2_RESULTS = ROOT / "results" / "phase2"
PHASE2_RESULTS.mkdir(parents=True, exist_ok=True)
print("Pinned QML environment check passed.")

Python 3.11.18
package                  required     installed        status
pennylane                0.44.1       0.44.1           OK
PennyLane-qiskit         0.44.1       0.44.1           OK
qiskit                   2.3.0        2.3.0            OK
qiskit-aer               0.17.2       0.17.2           OK
qiskit-ibm-runtime       0.45.1       0.45.1           OK
ibm-platform-services    0.77.1       0.77.1           OK
numpy                    2.4.6        2.4.6            OK
scipy                    1.17.1       1.17.1           OK
scikit-learn             1.9.1        1.9.1            OK
pandas                   3.1.0        3.1.0            OK
matplotlib               3.11.2       3.11.2           OK
jsonschema               4.26.0       4.26.0           OK
notebook                 7.6.3        7.6.3            OK
nbconvert                7.17.2       7.17.2           OK
ipykernel                7.4.0        7.4.0            OK
Pinned QML environment check passed.


## Task A — official Bell-state fidelity

Run the kit's unmodified `task_a_bell()` implementation at its official 8,192
shots per XX, YY, and ZZ basis. Save the complete returned record; fidelity is
`(1 + <XX> - <YY> + <ZZ>) / 4`, not the `00`/`11` population.

In [2]:
from benchmark.qff_qml_benchmark import task_a_bell

task_a_results = task_a_bell()
task_a_path = PHASE2_RESULTS / "task_a_results.json"
task_a_path.write_text(
    json.dumps(task_a_results, indent=2) + "\n",
    encoding="utf-8",
)
print(f"Saved official Task A results to {task_a_path.relative_to(ROOT)}")
print(json.dumps(task_a_results, indent=2))

Saved official Task A results to results/phase2/task_a_results.json
{
  "shots": 8192,
  "noise_model_version": "QFF2026-v2",
  "ideal": {
    "XX": 1.0,
    "XX_se": 0.0,
    "YY": -1.0,
    "YY_se": 0.0,
    "ZZ": 1.0,
    "ZZ_se": 0.0,
    "fidelity": 1.0,
    "fidelity_se": 0.0
  },
  "noisy": {
    "XX": 0.819091796875,
    "XX_se": 0.006338134717392098,
    "YY": -0.827880859375,
    "YY_se": 0.0061971720755083825,
    "ZZ": 0.87353515625,
    "ZZ_se": 0.005378000738451436,
    "fidelity": 0.880126953125,
    "fidelity_se": 0.0025920525815950387
  }
}


## Task B — official Processor D QFT-3 benchmark

Run the official command and preserve its stdout. The kit command exits with
status 0 even when embedded `benchmark_results` are stale, so this notebook
also parses the emitted JSON and compares every field with
`processor_d_config.json`. A warning-only mismatch is promoted to a hard
failure here.

In [3]:
benchmark_run = subprocess.run(
    [sys.executable, "benchmark/qff_qml_benchmark.py", "processor_d_config.json"],
    cwd=ROOT,
    check=False,
    capture_output=True,
    text=True,
)
benchmark_output = benchmark_run.stdout
if benchmark_output:
    print(benchmark_output, end="" if benchmark_output.endswith("\n") else "\n")
if benchmark_run.stderr:
    print(benchmark_run.stderr, file=sys.stderr, end="")
if benchmark_run.returncode != 0:
    raise RuntimeError(
        f"Official Task B command failed with exit code {benchmark_run.returncode}."
    )

task_b_path = PHASE2_RESULTS / "processor_d_benchmark.txt"
task_b_path.write_text(benchmark_output, encoding="utf-8")

marker = "Official QFT-3 benchmark_results:"
if marker not in benchmark_output:
    raise RuntimeError("Could not locate Task B benchmark_results in official stdout.")
emitted_payload = benchmark_output.split(marker, 1)[1].lstrip()
try:
    emitted_results, _ = json.JSONDecoder().raw_decode(emitted_payload)
except json.JSONDecodeError as error:
    raise RuntimeError("Could not parse emitted Task B benchmark JSON.") from error

processor_config = json.loads(
    (ROOT / "processor_d_config.json").read_text(encoding="utf-8")
)
configured_results = processor_config.get("benchmark_results")
if not isinstance(configured_results, dict):
    raise RuntimeError("processor_d_config.json has no benchmark_results object.")

all_result_keys = sorted(set(emitted_results) | set(configured_results))
task_b_mismatches = {
    key: {
        "emitted": emitted_results.get(key, "<missing>"),
        "configured": configured_results.get(key, "<missing>"),
    }
    for key in all_result_keys
    if emitted_results.get(key, "<missing>")
    != configured_results.get(key, "<missing>")
}
if task_b_mismatches:
    raise AssertionError(
        "Official Task B results differ from processor_d_config.json:\n"
        + json.dumps(task_b_mismatches, indent=2)
    )
print(
    "Task B emitted results exactly match processor_d_config.json; "
    f"saved stdout to {task_b_path.relative_to(ROOT)}."
)

Processor D design is valid. Official QFT-3 benchmark_results:
{
  "qft3_circuit_depth_before_transpile": 6,
  "qft3_circuit_depth_after_transpile": 18,
  "qft3_two_qubit_gate_count": 9,
  "qft3_swap_overhead": 0,
  "qft3_fidelity_ideal": 1.0,
  "qft3_fidelity_noisy": 0.668377,
  "benchmark_noise_model_version": "QFF2026-v2"
}
Task B emitted results exactly match processor_d_config.json; saved stdout to results/phase2/processor_d_benchmark.txt.


## PS-Q2 — six-qubit topology-aware quantum kernel

`run_experiment()` uses all 800 labeled rows and all 50 features. Within each
of five stratified folds, it fits standardization and six-component PCA only on
the fold's training partition. The fixed, shallow weak-ZZ feature map uses
`theta_scale=1.2`, `beta=0.1`, and Processor D's physical cycle
`4-0-2-5-1-3-4`. It has zero trainable quantum parameters; the six PCA values
are data inputs, not variational weights.

The all-feature RBF baseline retains all 50 standardized features and uses the
same rows and folds. Model selection never reads test labels. Final
preprocessing and fitting use all labeled rows before predictions are written.

In [4]:
from q2_solution import run_experiment

q2_results = run_experiment()
if not isinstance(q2_results, dict):
    raise TypeError("run_experiment() must return the generated metrics dictionary.")
q2_metrics_path = PHASE2_RESULTS / "q2_metrics.json"
if not q2_metrics_path.is_file():
    raise FileNotFoundError("run_experiment() did not write q2_metrics.json.")
print("PS-Q2 experiment completed and generated its metrics/predictions artifacts.")

PS-Q2 experiment completed and generated its metrics/predictions artifacts.


## Official prediction validation

Invoke the kit v3 validator after `run_experiment()` writes
`predictions_q2.csv`. A nonzero exit status or missing success message is a
notebook failure, not a warning.

In [5]:
validator_run = subprocess.run(
    [sys.executable, "tools/validate_predictions.py", "q2", "predictions_q2.csv"],
    cwd=ROOT,
    check=False,
    capture_output=True,
    text=True,
)
if validator_run.stdout:
    print(validator_run.stdout, end="" if validator_run.stdout.endswith("\n") else "\n")
if validator_run.stderr:
    print(validator_run.stderr, file=sys.stderr, end="")
if validator_run.returncode != 0:
    raise RuntimeError(
        "Official prediction validator failed with exit code "
        f"{validator_run.returncode}."
    )
if "OK - predictions file is valid." not in validator_run.stdout:
    raise RuntimeError("Official prediction validator did not emit its success message.")
print("Official PS-Q2 prediction validation passed.")

OK - predictions file is valid.
Official PS-Q2 prediction validation passed.


## Generated model, noise, confidence, and resource metrics

The display below reads `results/phase2/q2_metrics.json` and verifies that it
matches the object returned by `run_experiment()`, so copied values cannot
silently drift from the generated artifact. Final reporting
should retain every selected model's five fold scores, mean, sample standard
deviation, and descriptive 95% Student-*t* interval. These are training-only
cross-validation estimates, not hidden-test scores.

In [6]:
reported_results = json.loads(q2_metrics_path.read_text(encoding="utf-8"))
if reported_results != q2_results:
    raise RuntimeError("Returned metrics differ from the generated q2_metrics.json.")
model_selection = reported_results.get("model_selection", {})
diagnostics = reported_results.get("diagnostics", {})
fold_and_confidence = {
    key: value
    for key, value in diagnostics.items()
    if any(token in key.lower() for token in ("fold", "confidence", "interval", "std"))
}
for key, value in model_selection.items():
    if any(token in key.lower() for token in ("fold", "confidence", "interval", "balanced_accuracy")):
        fold_and_confidence.setdefault(key, value)

key_metrics = {
    "dataset": reported_results.get("dataset", {}),
    "model_selection": model_selection,
    "fold_and_confidence_reporting": fold_and_confidence,
    "quantum_kernel_and_noise": reported_results.get("quantum_kernel", {}),
    "noise_impact": reported_results.get("noise_impact", {}),
    "resource_cost": reported_results.get("resource_cost", {}),
    "prediction": {
        "file": reported_results.get("prediction_file"),
        "rows": reported_results.get("prediction_rows"),
    },
}
print(json.dumps(key_metrics, indent=2))

{
  "dataset": {
    "training_rows_available": 800,
    "training_rows_used": 800,
    "training_subset_rows": 800,
    "training_class_counts": {
      "A": 386,
      "B": 414
    },
    "test_rows": 200,
    "test_ids_preserved": true,
    "test_labels_available": false,
    "test_labels_used": false,
    "features_used_for_reduction": [
      "x01",
      "x02",
      "x03",
      "x04",
      "x05",
      "x06",
      "x07",
      "x08",
      "x09",
      "x10",
      "x11",
      "x12",
      "x13",
      "x14",
      "x15",
      "x16",
      "x17",
      "x18",
      "x19",
      "x20",
      "x21",
      "x22",
      "x23",
      "x24",
      "x25",
      "x26",
      "x27",
      "x28",
      "x29",
      "x30",
      "x31",
      "x32",
      "x33",
      "x34",
      "x35",
      "x36",
      "x37",
      "x38",
      "x39",
      "x40",
      "x41",
      "x42",
      "x43",
      "x44",
      "x45",
      "x46",
      "x47",
      "x48",
      "x49",
      "x50"
    ],


## Interpretation boundary

- Task A and Task B are seeded simulations under the organizer's fixed
  QFF2026-v2 conditions, not hardware measurements.
- The PS-Q2 noisy density-matrix kernel has no measurement step, so readout
  error does not enter that classifier calculation; Task A does include the
  official readout model.
- The cycle mapping avoids routing SWAPs for the ring interactions and limits
  two-qubit depth/noise relative to all-to-all connectivity, but also limits
  direct long-range expressivity.
- Zero Task B SWAP overhead is a topology result, not evidence of quantum
  classification advantage. Compare ideal, noisy, and RBF fold metrics
  directly and do not report hidden-test accuracy without organizer labels.